<a href="https://colab.research.google.com/github/MaazKhan53/ML-01-Run-the-Starter-Notebooks/blob/main/Copy_of_w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaazKhan53/ML-01-Run-the-Starter-Notebooks/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The queue prioritizes content by combining the model's predicted probability of decline with the page's historical volume. A high probability of decline on a zero-traffic page is mathematically irrelevant to the business. We map predictions to three specific reason codes to give human reviewers instant context:

Code 1 (Critical Refresh): High decline probability (>70%) + High historical traffic. Action: Immediate content refresh.

Code 2 (Monitor & Tweak): Medium decline probability (40-70%) + High traffic. Action: Technical SEO and meta-tag review.

Code 3 (Deprioritize): Low historical traffic regardless of probability. Action: Do nothing / Archive.

In [ ]:
import pandas as pd
import numpy as np

# Simulating loading our scored predictions from Week 5/6
# df = pd.read_csv('../outputs/scored_predictions.csv')
np.random.seed(42)
df = pd.DataFrame({
    'url_id': range(1, 101),
    'impressions': np.random.randint(100, 50000, 100),
    'decline_probability': np.random.uniform(0.1, 0.9, 100)
})

# Assign Reason Codes based on our playbook rules
def assign_action(row):
    if row['impressions'] > 10000 and row['decline_probability'] >= 0.70:
        return 'Code 1: Critical Refresh'
    elif row['impressions'] > 10000 and row['decline_probability'] >= 0.40:
        return 'Code 2: Monitor & Tweak'
    elif row['impressions'] < 2000:
        return 'Code 3: Deprioritize'
    else:
        return 'Standard Review'

df['action_code'] = df.apply(assign_action, axis=1)

# Sort the queue: highest probability and highest impressions first
action_queue = df.sort_values(by=['decline_probability', 'impressions'], ascending=[False, False])
action_queue.head()

,url_id,impressions,decline_probability,action_code
84,85,23347,0.888520,Code 1: Critical Refresh
69,70,45858,0.877426,Code 1: Critical Refresh
70,71,6973,0.869958,Standard Review
64,65,11753,0.854328,Code 1: Critical Refresh
42,43,10727,0.843758,Code 1: Critical Refresh


## 2. Intended use and limits

This model is strictly a directional, decision-support tool for content marketing teams. It relies on observed historical patterns and measured feature inputs (like CTR and position).

Limits: It predicts decline but does not prove causation. A high score means a page looks like pages that have historically dropped; it does not explain why it will drop. It is completely blind to external market forces (e.g., a competitor launching a new product today).

In [ ]:
# Check the distribution limits of our directional model
print(f"Total pages flagged for Code 1 (Critical): {len(action_queue[action_queue['action_code'] == 'Code 1: Critical Refresh'])}")
print(f"Base rate of predicted decline > 50%: {(len(action_queue[action_queue['decline_probability'] > 0.5]) / len(action_queue)) * 100:.1f}%")
print("Note: The model is bounded by its training horizon. Predictions beyond a 30-day window are unvalidated.")

Total pages flagged for Code 1 (Critical): 19
Base rate of predicted decline > 50%: 48.0%
Note: The model is bounded by its training horizon. Predictions beyond a 30-day window are unvalidated.


## 3. Human review + the no-go list

Every recommendation in the top queue requires a human in the loop. Reviewers must manually check for seasonality or intentional business pivots the model cannot see.

The No-Go List: We will never automate the unpublishing, redirecting, or rewriting of URLs based on this model. The business cost of a false positive (deleting a high-converting page) heavily outweighs the efficiency gained by automation.

In [ ]:
# Implementing the No-Go guardrail: Flagging highly sensitive pages (e.g., core product pages)
# In reality, this would merge with a CRM or CMS database of 'protected' URLs
protected_urls = [12, 45, 88] # Dummy IDs for core business pages

# Remove protected pages from the automated action queue to enforce human-only review
safe_queue = action_queue[~action_queue['url_id'].isin(protected_urls)].copy()

print(f"Filtered {len(action_queue) - len(safe_queue)} protected No-Go pages from the active queue.")

Filtered 3 protected No-Go pages from the active queue.


## 4. Monitoring / retrain triggers

Models degrade as the real world changes. To prevent silent failures, we monitor for Data Drift (changes in input distributions) and Prediction Drift (changes in output distributions).

Triggers: If the median average_position or impressions in our incoming live data shifts by more than 15% from our training baseline, or if the model suddenly predicts a decline rate 10% higher than historical averages, an alert is fired to pause the playbook and trigger an engineering review for retraining.

In [ ]:
# Simulating a basic Data Drift check on our primary feature (impressions)
training_mean_impressions = 25000 # Baseline from Week 5
current_mean_impressions = safe_queue['impressions'].mean()

drift_variance = abs(current_mean_impressions - training_mean_impressions) / training_mean_impressions

print(f"Current Impression Drift: {drift_variance:.2%}")
if drift_variance > 0.15:
    print("WARNING: Data drift threshold exceeded. Triggering retrain evaluation.")
else:
    print("Data distribution is stable within the 15% tolerance threshold.")

Current Impression Drift: 11.28%
Data distribution is stable within the 15% tolerance threshold.


## 5. Exports for the paper

We export the top 50 actionable items to the outputs folder. This CSV excludes any PII or sensitive client logic, acting purely as the operational queue for the research paper's findings.

In [ ]:
import os

# Ensure the output directory exists
os.makedirs('../outputs', exist_ok=True)

# Export the top 50 prioritized actions for the paper
final_export = safe_queue.head(50)
final_export.to_csv('../outputs/w07_action_queue.csv', index=False)

print("Successfully exported top 50 action queue to work/outputs/w07_action_queue.csv")

Successfully exported top 50 action queue to work/outputs/w07_action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.